# Improving TSP search with a search strategy

This short tutorial is for Qayd users who know the basics of constraint programming. It shows how a problem-specific search strategy can reduce the search tree without changing the model or its optimum.

By the end, you will know how to:

- model a small Traveling Salesperson Problem (TSP);
- define a `SearchPolicy`;
- compare the default and guided searches fairly.


## Outline

1. Define one small TSP instance.
2. Build the exact CP model.
3. Solve with Qayd's default search.
4. Solve with the Choco-style `max-regret/min` strategy.
5. Compare the search trees.


In [1]:
from dataclasses import dataclass
from math import hypot

import qayd as cp

CITIES = (
    (17, 62), (58, 29), (81, 69), (56, 57), (32, 27),
    (24, 65), (96, 31), (34, 77), (68, 51), (63, 16),
)

DISTANCES = tuple(
    tuple(round(hypot(x1 - x2, y1 - y2)) for x2, y2 in CITIES)
    for x1, y1 in CITIES
)

len(CITIES)


10

## 1. Build the model

For each city `i`, `successors[i]` is the next city in the tour and `edge_costs[i]` is the corresponding distance. `element_const` links those two variables. `circuit` requires one Hamiltonian cycle, and the objective minimizes the sum of the selected edge costs.


In [2]:
def build_model():
    city_count = len(DISTANCES)
    model = cp.Model()
    successors = model.int_vars(city_count, 0, city_count - 1, name="succ")
    edge_costs = model.int_vars(
        city_count, 0, max(map(max, DISTANCES)), name="cost"
    )
    for city in range(city_count):
        model.element_const(DISTANCES[city], successors[city], edge_costs[city])
    model.circuit(successors)
    model.minimize(cp.sum(edge_costs))
    return model, successors, edge_costs


The helper below rebuilds the model for every run, fixes `threads=1` and `seed=0`, and independently replays the returned tour. Only the search policy changes between the two arms.


In [3]:
@dataclass(frozen=True)
class Run:
    strategy: str
    objective: int
    nodes: int
    failures: int


def solve(variable_selector=None):
    model, successors, edge_costs = build_model()
    policy = None
    if variable_selector is not None:
        phase = cp.SearchPhase(edge_costs, variable_selector, "min")
        policy = cp.SearchPolicy([phase])

    solution = model.solve(
        engine="exact", search_policy=policy, threads=1, seed=0
    )
    assert solution.status == "OPTIMAL"

    selected = [solution.value(variable) for variable in successors]
    assert sorted(selected) == list(range(len(CITIES)))
    current, visited = 0, set()
    for _ in CITIES:
        assert current not in visited
        visited.add(current)
        current = selected[current]
    assert current == 0 and len(visited) == len(CITIES)

    replayed = sum(DISTANCES[city][selected[city]] for city in range(len(CITIES)))
    assert replayed == solution.objective
    return Run(
        variable_selector or "auto", replayed,
        solution.stats.nodes, solution.stats.failures,
    )


## 2. Baseline: default search

With no explicit policy, Qayd uses its general-purpose automatic search.


In [4]:
auto = solve()
auto


Run(strategy='auto', objective=244, nodes=220, failures=125)

## 3. Guided search: `max-regret` then `min`

The [Choco TSP tutorial](https://choco-solver.org/tutos/traveling-salesman-problem/code/) branches on the edge-cost variables. `max-regret` selects the variable with the largest gap between its two smallest supported values, then `min` tries the smallest value first. Intuitively, the search handles expensive missed opportunities early.

The phase covers only the edge costs. Qayd's implicit `Auto` fallback still assigns any remaining successor variables, so the exact search remains complete.


In [5]:
guided = solve("max-regret")
guided


Run(strategy='max-regret', objective=244, nodes=34, failures=27)

## 4. Compare the results

Both searches must prove the same optimum. The interesting difference is how much search work they perform.


In [6]:
assert auto.objective == guided.objective == 244
reduction = 100 * (auto.nodes - guided.nodes) / auto.nodes

print(f"Auto:       {auto.nodes} nodes, {auto.failures} failures")
print(f"MaxRegret:  {guided.nodes} nodes, {guided.failures} failures")
print(f"Node reduction: {reduction:.1f}%")


Auto:       220 nodes, 125 failures
MaxRegret:  34 nodes, 27 failures
Node reduction: 84.5%


On this instance, the guided search reduces the tree from 220 to 34 nodes, a reduction of 84.5%, while proving the same optimum. This is a focused demonstration, not a claim that one strategy dominates on every TSP.

### Common pitfalls

- Compare fresh copies of the same model.
- Keep the seed and thread count identical.
- Compare objective and validity before comparing speed or node counts.
- Node counts can change when the solver implementation changes, so rerun the notebook after upgrades.


## Exercise

Replace `max-regret` with `first-fail`. Does it improve on Auto? Does it improve as much as `max-regret`? Predict the result before running the next cell.


In [7]:
first_fail = solve("first-fail")
print(first_fail)
print(f"Improves on Auto: {first_fail.nodes < auto.nodes}")
print(f"Beats MaxRegret: {first_fail.nodes < guided.nodes}")


Run(strategy='first-fail', objective=244, nodes=146, failures=101)
Improves on Auto: True
Beats MaxRegret: False
